# Desafio Técnico Keyrus — Bank Marketing

Data Scientist — desafio de modelagem em Python.

**Objetivo:** prever se um cliente vai assinar um depósito a prazo (`y`), a partir do dataset [Bank Marketing](https://www.kaggle.com/datasets/abdelazizsami/bank-marketing/data) (Kaggle), réplica do dataset público UCI de Moro et al. (2011).

Roteiro seguido (ver `docs/Teste_Modelagem_Python.pdf`): EDA → pré-processamento → modelagem → avaliação → interpretação → uso em produção.

## 0. Configuração e carregamento dos dados

Leitura direta do Kaggle via [`kagglehub`](https://github.com/Kaggle/kagglehub), sem baixar/versionar os CSVs no repositório.

O dataset traz dois arquivos (ver `docs/bank-names.txt`):
- `bank-full.csv`: todos os 45.211 exemplos.
- `bank.csv`: amostra de 10% (4.521 exemplos).

**Pré-requisito:** credenciais do Kaggle configuradas (`~/.kaggle/kaggle.json` ou variável `KAGGLE_API_TOKEN`). Sem isso, o download abaixo falha na autenticação.

In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd

RANDOM_SEED = 42  # usada em todas as etapas do notebook que envolvam aleatoriedade

DATASET = "abdelazizsami/bank-marketing"

In [2]:
# O dataset original (Moro et al., 2011) usa ';' como separador de campos.
bank_full_df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    DATASET,
    "bank-full.csv",
    pandas_kwargs={"sep": ";"},
)

bank_df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    DATASET,
    "bank.csv",
    pandas_kwargs={"sep": ";"},
)

# 16 atributos de entrada + 1 alvo (y) = 17 colunas esperadas.
for nome, df in [("bank-full.csv", bank_full_df), ("bank.csv", bank_df)]:
    assert df.shape[1] == 17, (
        f"{nome}: esperadas 17 colunas, encontradas {df.shape[1]} — "
        "verifique o separador do CSV."
    )
    print(f"{nome}: {df.shape[0]} linhas, {df.shape[1]} colunas")

bank-full.csv: 45211 linhas, 17 colunas
bank.csv: 4521 linhas, 17 colunas


In [3]:
bank_full_df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [4]:
bank_df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,199,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,226,1,-1,0,unknown,no
